In [6]:
import os
from dotenv import load_dotenv
import streamlit as st
import pickle
import time
import langchain

from langchain_openai import OpenAI
from langchain_classic.chains import RetrievalQAWithSourcesChain
from langchain_classic.chains.qa_with_sources.loading import load_qa_with_sources_chain
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import UnstructuredURLLoader

from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS

load_dotenv()

True

In [7]:
from langchain_google_genai import ChatGoogleGenerativeAI
llm= ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0.7)


In [10]:
url_loader = UnstructuredURLLoader(urls=[
        "https://www.moneycontrol.com/news/business/banks/rbi-allows-banks-to-offer-differential-interest-rates-on-bulk-deposits-13989824.html",
        "https://www.moneycontrol.com/news/business/information-technology/you-need-companies-like-infosys-to-make-ai-work-in-enterprises-says-ceo-parekh-13990031.html"
])

data = url_loader.load()
len(data)

2

In [12]:
text_splitter = RecursiveCharacterTextSplitter(
    chunk_size = 1000,
    chunk_overlap = 200
)
docs = text_splitter.split_documents(data)
len(docs)

22

In [13]:
docs[1]

Document(metadata={'source': 'https://www.moneycontrol.com/news/business/banks/rbi-allows-banks-to-offer-differential-interest-rates-on-bulk-deposits-13989824.html'}, page_content='Eco Pulse\n\nMC Learn\n\nPersonal Loan Offers\n\nPersonal Loan Offers\n\nTrending Topics\n\nSensex Live\n\nITR Filing Last Date\n\nJuniper Green Energy IPO\n\nManipal Health IPO\n\nGold and Silver Prices Today\n\nRBI allows banks to offer differential interest rates on bulk deposits\n\nThe move comes after the HDFC Bank-MSRDC controversy, where the private lender allegedly offered above-market interest rates to secure deposits\n\nArchishma Iyer\n\nJuly 31, 2026 / 10:18 IST\n\njoin Us On WhatsApp\n\nFollow Us On Google\n\nAdd as a Preferred Source on Google\n\n\n\nReserve Bank of India\n\nThe Reserve Bank of India (RBI) has allowed banks to offer different interest rates on bulk deposits based on their specific run-off rates under the liquidity coverage ratio (LCR) framework.\n\nThe rule will be applicable fo

In [14]:
embeddings = OpenAIEmbeddings()
vectorindex_openai = FAISS.from_documents(docs, embeddings)

RateLimitError: Error code: 429 - {'error': {'message': 'You have no credits remaining. Add credits to continue using the API at https://platform.openai.com/settings/organization/billing/.', 'type': 'insufficient_quota', 'param': None, 'code': 'credit_balance_exhausted'}}

In [15]:
from langchain_huggingface import HuggingFaceEmbeddings
# from langchain_chroma import Chroma

embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-V2")
# vector_store = Chroma.from_documents(chunks, embeddings)
vectorindex_openai = FAISS.from_documents(docs, embeddings)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 6113.45it/s]


In [16]:
file_path = "vector_index.pkl"
with open(file_path, "wb") as f:
    pickle.dump(vectorindex_openai, f)

In [19]:
if os.path.exists(file_path):
    with open(file_path, "rb") as f:
        vector_index = pickle.load(f)


chain = RetrievalQAWithSourcesChain.from_llm(llm = llm, retriever = vector_index.as_retriever())
chain

RetrievalQAWithSourcesChain(verbose=False, combine_documents_chain=MapReduceDocumentsChain(verbose=False, llm_chain=LLMChain(verbose=False, prompt=PromptTemplate(input_variables=['context', 'question'], input_types={}, partial_variables={}, template='Use the following portion of a long document to see if any of the text is relevant to answer the question.\nReturn any relevant text verbatim.\n{context}\nQuestion: {question}\nRelevant text, if any:'), llm=ChatGoogleGenerativeAI(metadata={'lc_versions': {'langchain-core': '1.4.8', 'langchain': '1.3.11', 'langchain-google-genai': '4.2.6'}}, output_version=None, profile={'name': 'Gemini 2.5 Flash', 'release_date': '2025-03-20', 'last_updated': '2025-06-05', 'open_weights': False, 'max_input_tokens': 1048576, 'max_output_tokens': 65536, 'text_inputs': True, 'image_inputs': True, 'audio_inputs': True, 'pdf_inputs': True, 'video_inputs': True, 'text_outputs': True, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reason

In [20]:
question1 = "by at what time banks displays interest rates?"
chain({"question": question}, return_only_outputs=True)

C:\Users\KOTI\AppData\Local\Temp\ipykernel_16024\1474997166.py:2: LangChainDeprecationWarning: The method `Chain.__call__` was deprecated in langchain-classic 0.1.0 and will be removed in 2.0.0. Use `invoke` instead.
  chain({"question": question}, return_only_outputs=True)


{'answer': 'All banks should display bulk deposit interest rates by 10 am every day, with a grace period of 10 minutes.\n',
 'sources': 'https://www.moneycontrol.com/news/business/banks/rbi-allows-banks-to-offer-differential-interest-rates-on-bulk-deposits-13989824.html'}

In [21]:
query = "When Parekh is going to retire"
chain({"question": query}, return_only_outputs=True)

{'answer': 'Parekh is going to retire on March 31, 2027.\n',
 'sources': 'https://www.moneycontrol.com/news/business/information-technology/you-need-companies-like-infosys-to-make-ai-work-in-enterprises-says-ceo-parekh-13990031.html'}

In [22]:
# Another way - skipping pickle storing and retrieving part above.

# 1. Generate embeddings and create the FAISS index in memory
embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-V2")
vectorindex_openai = FAISS.from_documents(docs, embeddings)

# 2. Skip pickle completely and pass the memory object directly into the chain
chain = RetrievalQAWithSourcesChain.from_llm(llm=llm, retriever=vectorindex_openai.as_retriever())

# 3. Query your model
query = "When Parekh is going to retire"
chain({"question": query}, return_only_outputs=True)

Loading weights: 100%|██████████| 103/103 [00:00<00:00, 2758.85it/s]


{'answer': 'Parekh will step down on March 31, 2027.\n',
 'sources': 'https://www.moneycontrol.com/news/business/information-technology/you-need-companies-like-infosys-to-make-ai-work-in-enterprises-says-ceo-parekh-13990031.html'}

In [23]:
# retriever=vectorindex_openai.as_retriever(search_kwargs={"k":3})
# retrieved = retriever.invoke(query)
# retrieved

[Document(id='bd61cd8d-14e7-4c7f-ac66-e71cc9f762b3', metadata={'source': 'https://www.moneycontrol.com/news/business/information-technology/you-need-companies-like-infosys-to-make-ai-work-in-enterprises-says-ceo-parekh-13990031.html'}, page_content='Parekh, who will step down on March 31, 2027 after more than nine years at the helm, said the company has already embedded AI into its delivery model, with about 80,000 employees using coding tools across client engagements and internal projects.\n\nDespite AI-led productivity gains, Infosys is not slowing recruitment.\n\nThe company hired 20,000 college graduates last year and plans to recruit a similar number this year, with more than 4,000 already onboarded in the first quarter.\n\nRelated Stories\n\n"Leaving Infosys in a better place": CEO Salil Parekh on AI and succession\n\nI\'m leaving Infosys in a better place: Outgoing CEO Salil Parekh says company is poised to win in AI...\n\nWhen asked if the company was caught the AI wave late, 